# Lesson 4: Persistence and Streaming

### 本节课的核心思路：持久化（Persistence）与流式输出（Streaming）

这一课在 Lesson 2 的 Agent 基础上加了两个能力：

- **Persistence / Checkpointer（检查点持久化）**：给 `graph.compile()` 传入一个 `checkpointer`，
  LangGraph 会在每一步之后自动把 state 存下来，并用 `thread_id` 区分不同的会话。
  这样同一个 `thread_id` 的多轮对话之间就能"记住"之前的消息（不需要自己维护 messages 列表拼接），
  不同 `thread_id` 之间则完全隔离，互不干扰——这是构建多用户/多会话 Agent 服务的基础。
- **Streaming（流式输出）**：`graph.stream(...)` 按节点执行的进度逐步吐出中间结果，
  `graph.astream_events(...)` 更进一步，能拿到模型 token 级别的流式增量，实现打字机效果。

In [ ]:
from dotenv import load_dotenv

_ = load_dotenv()

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults  # 旧版路径，当前仍可用（新版推荐 langchain_tavily.TavilySearch）

In [ ]:
tool = TavilySearchResults(max_results=2)

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]  # operator.add 让每轮新消息"累加"进历史，而不是覆盖

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver
# 注：SqliteSaver 是独立发布的包 langgraph-checkpoint-sqlite（本机 venv 已额外装上，课程原始环境里可能是内置的旧版）

# 【版本变化修复】旧版 langgraph 里 SqliteSaver.from_conn_string(":memory:") 直接返回一个可用的 SqliteSaver 实例，
# 但当前安装的新版把它实现成了一个上下文管理器（@contextmanager），如果像原代码一样直接
#     memory = SqliteSaver.from_conn_string(":memory:")
# 拿到的其实是一个 _GeneratorContextManager 对象，并不是真正的 SqliteSaver，
# 后面把它当 checkpointer 传给 graph.compile() 后，图在真正读写检查点时会报
# AttributeError: '_GeneratorContextManager' object has no attribute 'get_tuple'。
# 正确用法是 `with SqliteSaver.from_conn_string(...) as memory:`，但这样后面所有用到 memory 的 cell
# 都要嵌套在同一个 with 块里，不方便在 notebook 里跨 cell 使用。
# 这里手动调用 .__enter__() 拿到真正的 SqliteSaver 实例，效果等价于进入了 with 块；
# 同时必须把上下文管理器本身也保存到一个变量（_memory_cm）里长期持有——
# 如果只保留 .__enter__() 的返回值、不保留上下文管理器对象本身，
# 它会在某次垃圾回收时被当成"没人用了"而自动触发退出逻辑，把底层的 sqlite 连接关掉，
# 导致后面的 cell 报 "Cannot operate on a closed database."。
_memory_cm = SqliteSaver.from_conn_string(":memory:")
memory = _memory_cm.__enter__()

In [ ]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)   # 关键区别：这里把 checkpointer 传给 compile()，
        # 这样每次 graph.invoke/stream 时带上 thread_id，LangGraph 就会自动读写这个 checkpointer 里保存的状态
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)  # 把上面创建的 SqliteSaver 作为 checkpointer 传进去

In [ ]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [ ]:
thread = {"configurable": {"thread_id": "1"}}  # thread_id 是这次会话的唯一标识，checkpointer 靠它来存取对应的历史状态

In [ ]:
for event in abot.graph.stream({"messages": messages}, thread):   # stream 按节点执行进度逐步产出中间结果（而不是等全部跑完才返回）
    for v in event.values():
        print(v['messages'])

In [ ]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}   # 同一个 thread_id="1"：checkpointer 会自动带上之前的对话历史，
# 所以模型能理解 "in la" 这个省略主语的追问其实还是在问天气
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

In [ ]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}   # 继续用 thread_id="1"，模型能看到前两轮 sf/la 的天气结果来做比较
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

In [ ]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}   # 换成一个全新的 thread_id="2"：checkpointer 里没有这个会话的历史，
# 模型看不到之前 sf/la 的天气信息，应该会反问"warmer than what?"，用来验证不同 thread 之间状态是隔离的
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

In [ ]:
# 【版本变化修复】原代码写的是 `from langgraph.checkpoint.aiosqlite import AsyncSqliteSaver`，
# 这个模块路径在当前安装的 langgraph-checkpoint-sqlite 新版里已经不存在了（ModuleNotFoundError），
# AsyncSqliteSaver 现在放在 langgraph.checkpoint.sqlite.aio 子模块下，改成下面这样导入。
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

# 同 SqliteSaver 一样，from_conn_string(":memory:") 返回的是一个异步上下文管理器（async context manager），
# 不能直接当 checkpointer 用；这里用 await ... .__aenter__() 手动"进入"上下文，拿到真正的 AsyncSqliteSaver 实例。
# 同样要把上下文管理器对象（_async_memory_cm）单独保存下来，避免被垃圾回收提前关闭底层连接
# （Jupyter 单元格支持顶层 await，所以可以直接写 await 而不用包一层 async def）。
_async_memory_cm = AsyncSqliteSaver.from_conn_string(":memory:")
memory = await _async_memory_cm.__aenter__()
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [ ]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
# astream_events：比 stream 粒度更细的流式接口，能拿到模型生成过程中每个 token 级别的增量事件
# version="v1" 是旧版事件 schema（当前 langchain-core 默认是 "v2"，但 "v1" 仍受支持，不会报错）
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")